In [ ]:
# 필요 라이브러리 불러오기

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 데이터 분할, 표준화, 과적합 방지 관련 도구들
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, average_precision_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 120.7 kB/s  0:00:20 eta 0:00:03


In [4]:
# 1. 데이터 로드 및 기본 탐색

df = pd.read_csv('creditcard.csv') # 다운받은 신용카드 데이터셋을 불러옵니다.

# 데이터 상위 5개 행 확인해서 컬럼 종류 파악
print("--- Data Head ---")
print(df.head())

# 데이터의 전체적인 정보 (행/열 개수, 데이터 타입, 결측치 유무) 확인
print("\n--- Data Info ---")
print(df.info())

# Class 컬럼: 0은 정상 거래, 1은 사기 거래
# 사기 거래(1)와 정상 거래(0)의 비율 및 건수를 확인합니다.
print("\n--- Class Ratio (Original) ---")
print(df['Class'].value_counts(normalize=True)) # 비율로 출력
print(df['Class'].value_counts()) # 실제 건수로 출력

--- Data Head ---
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26   

In [5]:
# 2. 샘플링

# 사기 데이터(Class=1)는 수가 적으므로 전부(492건) 다 가져옵니다.
fraud_df = df[df['Class'] == 1]

# 정상 데이터(Class=0)는 용량이 커서 명세서 지시대로 10,000건만 무작위 추출합니다. (random_state=42로 고정)
normal_df = df[df['Class'] == 0].sample(n=10000, random_state=42)

# 추출한 두 데이터를 하나로 합친 뒤 순서가 섞이도록 셔플(sample(frac=1))해줍니다.
df_sampled = pd.concat([fraud_df, normal_df]).sample(frac=1, random_state=42).reset_index(drop=True)

# 샘플링 완료 후 Class 비율이 어떻게 바뀌었는지 다시 확인합니다.
print("\n--- Sampled Class Ratio ---")
print(df_sampled['Class'].value_counts(normalize=True))
print(df_sampled['Class'].value_counts())


--- Sampled Class Ratio ---
Class
0    0.953107
1    0.046893
Name: proportion, dtype: float64
Class
0    10000
1      492
Name: count, dtype: int64


In [6]:
# 3. 데이터 전처리

# Amount(거래 금액) 변수는 숫자의 단위가 매우 크기 때문에 표준화를 진행합니다.
scaler = StandardScaler()
df_sampled['Amount_Scaled'] = scaler.fit_transform(df_sampled[['Amount']])

# 스케일링 전 원본 Amount 변수는 필요 없으므로 제거해줍니다.
df_sampled = df_sampled.drop(columns=['Amount'])

# 모델 학습에 사용할 독립변수(X)와 정답 레이블인 타겟변수(y)로 분리합니다.
X = df_sampled.drop(columns=['Class'])  # Class를 제외한 나머지 모든 컬럼
y = df_sampled['Class'] # Class 컬럼 (0 또는 1)

In [7]:
# 4. 학습 데이터와 테스트 데이터 분할

# train_test_split을 사용하여 학습용 80%, 테스트용 20% 비율로 나눕니다.
# stratify=y 옵션을 주어 Train과 Test 셋 안에서 Class(0과 1) 비율이 동일하게 유지되도록 합니다.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\n--- Train & Test Class Distribution ---")
print("Train Class Ratio:\n", y_train.value_counts(normalize=True))
print("Test Class Ratio:\n", y_test.value_counts(normalize=True))


--- Train & Test Class Distribution ---
Train Class Ratio:
 Class
0    0.953056
1    0.046944
Name: proportion, dtype: float64
Test Class Ratio:
 Class
0    0.953311
1    0.046689
Name: proportion, dtype: float64


In [9]:
# 5. SMOTE 적용
# SMOTE 적용 이유 : 
# 정상 거래(0)에 비해 사기 거래(1) 건수가 훨씬 적기 때문에, 모델이 무조건 정상이라고 예측해버리는 편향 문제가 생깁니다.
# 따라서 소수 클래스(사기 거래)의 데이터를 가상으로 합성하여 늘려주는 SMOTE 기법을 학습 데이터(X_train)에만 적용합니다.
print("\nSMOTE 적용 전 y_train 개수:", y_train.value_counts().to_dict())

# SMOTE 객체 생성 후 학습 데이터에 적용 (random_state=42)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("SMOTE 적용 후 y_train 개수:", y_train_res.value_counts().to_dict())


SMOTE 적용 전 y_train 개수: {0: 7999, 1: 394}
SMOTE 적용 후 y_train 개수: {0: 7999, 1: 7999}


In [10]:
# 6. 모델 학습 & 예측 (XGBoost 사용)

# 불균형 데이터 분류 성능이 우수한 XGBoost 분류기 모델을 생성하고 학습시킵니다.
model = XGBClassifier(
    n_estimators=200,  # 결정 트리 개수 200개
    learning_rate=0.05, # 학습률 0.05
    max_depth=4, # 트리 최대 깊이 4 (과적합 방지)
    random_state=42, 
    scale_pos_weight=1
)
# SMOTE로 오버샘플링된 학습 데이터를 사용하여 모델 학습 진행
model.fit(X_train_res, y_train_res)

# 테스트 데이터셋을 이용해 0/1 예측값과 사기(1)일 확률값을 각각 구합니다.
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# 기본 임계값(0.5) 기준 분류 리포트 출력 (Precision, Recall, F1-score 확인)
print("\n--- Classification Report (Default Threshold 0.5) ---")
print(classification_report(y_test, y_pred, digits=4))

# PR-AUC 점수 계산 및 출력
pr_auc = average_precision_score(y_test, y_pred_proba)
print(f"PR-AUC Score: {pr_auc:.4f}")


--- Classification Report (Default Threshold 0.5) ---
              precision    recall  f1-score   support

           0     0.9925    0.9935    0.9930      2001
           1     0.8646    0.8469    0.8557        98

    accuracy                         0.9867      2099
   macro avg     0.9285    0.9202    0.9243      2099
weighted avg     0.9865    0.9867    0.9866      2099

PR-AUC Score: 0.9089


In [11]:
# 7. Threshold 조정 및 최종 성능 평가
# 임계값(Threshold) 조정을 통한 Precision, Recall, F1-score 최적화
custom_threshold = 0.7  # 임계값 조정 (0.5 -> 0.7)
y_pred_custom = (y_pred_proba >= custom_threshold).astype(int)

print(f"\n--- Classification Report (Custom Threshold {custom_threshold}) ---")
report = classification_report(y_test, y_pred_custom, digits=4)
print(report)

# 최종 PR-AUC 값 출력
pr_auc_final = average_precision_score(y_test, y_pred_proba)
print(f"Final PR-AUC Score: {pr_auc_final:.4f}")


--- Classification Report (Custom Threshold 0.7) ---
              precision    recall  f1-score   support

           0     0.9920    0.9970    0.9945      2001
           1     0.9318    0.8367    0.8817        98

    accuracy                         0.9895      2099
   macro avg     0.9619    0.9169    0.9381      2099
weighted avg     0.9892    0.9895    0.9893      2099

Final PR-AUC Score: 0.9089
